In [1]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import os

from specula.mmlib.yaml_overrides import write_yaml_overrides
from specula.mmlib.compute_rec import compute_and_save_rec

2026-10-30 16:35:39,882 [INFO]: [specula]: Cupy import successfull. Installed version is: 13.3.0
2026-10-30 16:35:40,079 [INFO]: [specula]: Default device is GPU number 0


In [2]:
import matplotlib as mpl
from cycler import cycler

cmap = mpl.colormaps['hot'] # 1. Choose your colormap (e.g., 'hot', 'jet', 'turbo')
colors = cmap(np.linspace(0.0, 1.0, 7)) # 2. Extract a set of colors from it (e.g., 6 distinct colors across the map)
# plt.rcParams['axes.prop_cycle'] = cycler(color=colors) # 3. Apply it to the default line cycle
plt.rcParams['image.cmap'] = 'magma' # 4. Change the default colormap for images/scatter plots
plt.rcParams.update({
    "text.usetex": True, # Use LaTeX to write all text
    "font.family": "serif", # Use a serif font
    "font.serif": ["Computer Modern Roman"] # Specifically use the classic LaTeX font
})
plt.rcParams['image.origin'] = 'lower'

In [3]:
nSubaps = [10,14,24,36,48]
rMods = [0,0.5,1,2]
nModes = [54,100,300,800,1200,1600]
wvls = [900,1300]
overrides_yaml = 'temp_overrides'

calib_dir = '/raid1/mmenessini/calibration/WFSW'
result_dir = '/raid1/mmenessini/results/WFSW'

In [8]:
# Generate pupil data
for nSubap in nSubaps:
    tag = f'pyr_pupdata_{nSubap}x{nSubap}'
    try:
        pupdata = fits.getdata(os.path.join(calib_dir,'pupils',tag+'.fits'))
    except FileNotFoundError:
        ovdes = ("{"
            f"pyr_vis.pup_diam: {nSubap}, "
            f"pyr_vis.pup_dist: {np.max((14,np.ceil(nSubap*60/48)))}, "
            f"pyr_pupdata.output_tag: {tag}, "
            "}")
        write_yaml_overrides(ovdes,temp_name=overrides_yaml)
        os.system(f'specula ristretto_main.yml calib_pupdata.yml {overrides_yaml}.yml')

2026-10-30 16:39:02,475 [INFO]: [specula]: Cupy import successfull. Installed version is: 13.3.0
2026-10-30 16:39:02,714 [INFO]: [specula]: Default device is GPU number 0
2026-10-30 16:39:02,801 [INFO]: [specula.simul]: Reading parameters from ristretto_main.yml
2026-10-30 16:39:02,820 [INFO]: [specula.simul]: Reading additional parameters from calib_pupdata.yml
2026-10-30 16:39:02,822 [INFO]: [specula.simul]: Removed atmo
2026-10-30 16:39:02,822 [INFO]: [specula.simul]: Deleted {'atmo.layer_list'} from {'atmo_layer_list': ['atmo.layer_list'], 'common_layer_list': ['pupilstop', 'dm.out_layer:-1']}
2026-10-30 16:39:02,822 [INFO]: [specula.simul]: Removed seeing
2026-10-30 16:39:02,822 [INFO]: [specula.simul]: Removed wind_direction
2026-10-30 16:39:02,822 [INFO]: [specula.simul]: Removed wind_speed
2026-10-30 16:39:02,823 [INFO]: [specula.simul]: Removed dm
2026-10-30 16:39:02,823 [INFO]: [specula.simul]: Deleted {'dm.out_layer:-1'} from {'atmo_layer_list': [], 'common_layer_list': ['pu

220 440
220 440


2026-10-30 16:39:03,367 [INFO]: [prop]: cupy memory used by AtmoPropagation: 1.84912109375 MB
2026-10-30 16:39:03,413 [INFO]: [pyr_vis]: Cached circular modulation with 240 steps, amplitude: 30.00
2026-10-30 16:39:03,522 [INFO]: [pyr_vis]: cupy memory used by ModulatedPyramid: 901.4794921875 MB
2026-10-30 16:39:03,527 [INFO]: [pyr_nir]: Cached circular modulation with 24 steps, amplitude: 3.00
2026-10-30 16:39:03,539 [INFO]: [pyr_nir]: cupy memory used by ModulatedPyramid: 103.87109375 MB
2026-10-30 16:39:03,539 [INFO]: [cred_vis]: cupy memory used by CCD: 20.525390625 MB
2026-10-30 16:39:03,539 [INFO]: [cred_nir]: cupy memory used by CCD: 20.525390625 MB
2026-10-30 16:39:03,539 [INFO]: [pyr_pupdata]: cupy memory used by PyrPupdataCalibrator: 0.0 MB


Pupil 0: radius = 7.2, center = [50.5 50.5]
Pupil 1: radius = 7.2, center = [ 8.5 50.5]
Pupil 2: radius = 7.2, center = [50.5  8.5]
Pupil 3: radius = 7.2, center = [8.5 8.5]


2026-10-30 16:39:03,880 [INFO]: [specula.LoopControl]: t=0.005000 29.33 Hz,  34.097 ms
2026-10-30 16:39:03,938 [INFO]: [pyr_pupdata]: Saved pupil data: /raid1/mmenessini/calibration/WFSW/pupils/pyr_pupdata_14x14.fits
2026-10-30 16:39:03,938 [INFO]: [pyr_pupdata]: Obstruction ratio: 0.160


In [ ]:
# Generate IM data
for wvl in wvls:
    for rMod in rMods:
        for nSubap in nSubaps:
            tag = f'pyr{rMod:1.1f}_{nSubap:1.0f}x{nSubap:1.0f}_wvl{wvl:1.0f}_bmc2k_vlt'
            im_tag = tag+'_im'
            try:
                pupdata = fits.getdata(os.path.join(calib_dir,'pupils',tag+'.fits'))
            except FileNotFoundError:
                print(f'Simulating: {nSubap}x{nSubap}, {wvl}nm, {rMod:1.1f}'+r' $\lambda$/D')
                ovdes = ("{"
                    f"pyr_vis.pup_diam: {nSubap}, "
                    f"pyr_vis.pup_dist: {np.max((14,np.ceil(nSubap*60/48)))}, "
                    f"source_vis.wavelengthInNm: {wvl}, "
                    f"pyr_vis.mod_amp: {rMod:1.1f}, "
                    f"pyr_im_calibrator.im_tag: {im_tag}, "
                    f"pyr_vis.wavelengthInNm: {wvl}, "
                    f"pyr_slopes_vis.pupdata_object: pyr_pupdata_{nSubap}x{nSubap}, "
                    "}")
                write_yaml_overrides(ovdes,temp_name=overrides_yaml)
                os.system(f'specula ristretto_main.yml calib_im.yml {overrides_yaml}.yml')

                for N in nModes:
                    rec_tag = tag+f'_{N:1.0f}modes_rec'
                    compute_and_save_rec(root_dir=calib_dir, im_tag=im_tag, rec_tag=rec_tag, Nmodes=N)

Simulating: 10x10, 900nm, 0.0 $\lambda$/D


2026-10-30 16:39:08,135 [INFO]: [specula]: Cupy import successfull. Installed version is: 13.3.0
2026-10-30 16:39:08,340 [INFO]: [specula]: Default device is GPU number 0
2026-10-30 16:39:08,426 [INFO]: [specula.simul]: Reading parameters from ristretto_main.yml
2026-10-30 16:39:08,446 [INFO]: [specula.simul]: Reading additional parameters from calib_im.yml
2026-10-30 16:39:08,449 [INFO]: [specula.simul]: Removed atmo
2026-10-30 16:39:08,449 [INFO]: [specula.simul]: Removed seeing
2026-10-30 16:39:08,449 [INFO]: [specula.simul]: Removed wind_speed
2026-10-30 16:39:08,449 [INFO]: [specula.simul]: Removed wind_direction
2026-10-30 16:39:08,449 [INFO]: [specula.simul]: Removed atmo_modes
2026-10-30 16:39:08,449 [INFO]: [specula.simul]: Deleted {'atmo_res-atmo_modes.out_value'} from {'input_list': ['sr_vis-coro_psf_vis.out_sr', 'coro_psf_vis_int-coro_psf_vis.out_int_coronagraph_psf_profile', 'coro_psf_vis_std-coro_psf_vis.out_std_coronagraph_psf_profile', 'sr_nir-coro_psf_nir.out_sr', 'cor

220 440


2026-10-30 16:39:09,108 [INFO]: [specula.simul]: Restoring: /raid1/mmenessini/calibration/WFSW/pupils/pyr_pupdata_10x10.fits
2026-10-30 16:39:09,112 [INFO]: [specula.PupData]: cupy memory used by PupData: 0.0029296875 MB
2026-10-30 16:39:09,128 [INFO]: [specula.lib.calc_geometry]: FoV reduction from 7.19 to 2.00 will be performed with a focal plane mask
/raid1/mmenessini/git/SPECULA/specula/processing_objects/ccd.py:183: RuntimeWarning: ATTENTION: emccd_gain will not be used if excess_noise is False
  warnings.warn('ATTENTION: emccd_gain will not be used if excess_noise is False',
2026-10-30 16:39:09,136 [INFO]: [specula.simul]: Restoring: /raid1/mmenessini/calibration/WFSW/pupils/pyr_pupdata_48x48.fits
2026-10-30 16:39:09,139 [INFO]: [specula.PupData]: cupy memory used by PupData: 0.05517578125 MB
2026-10-30 16:39:09,150 [INFO]: [specula.simul]: Restoring: /raid1/mmenessini/calibration/WFSW/ifunc/bmc2k_vlt_ifunc.fits


220 440


...

2026-10-30 16:43:26,421 [INFO]: [specula]: Cupy import successfull. Installed version is: 13.3.0
2026-10-30 16:43:26,634 [INFO]: [specula]: Default device is GPU number 0
2026-10-30 16:43:26,720 [INFO]: [specula.simul]: Reading parameters from ristretto_main.yml
2026-10-30 16:43:26,739 [INFO]: [specula.simul]: Reading additional parameters from calib_im.yml
2026-10-30 16:43:26,742 [INFO]: [specula.simul]: Removed atmo
2026-10-30 16:43:26,742 [INFO]: [specula.simul]: Removed seeing
2026-10-30 16:43:26,742 [INFO]: [specula.simul]: Removed wind_speed
2026-10-30 16:43:26,742 [INFO]: [specula.simul]: Removed wind_direction
2026-10-30 16:43:26,742 [INFO]: [specula.simul]: Removed atmo_modes
2026-10-30 16:43:26,742 [INFO]: [specula.simul]: Deleted {'atmo_res-atmo_modes.out_value'} from {'input_list': ['sr_vis-coro_psf_vis.out_sr', 'coro_psf_vis_int-coro_psf_vis.out_int_coronagraph_psf_profile', 'coro_psf_vis_std-coro_psf_vis.out_std_coronagraph_psf_profile', 'sr_nir-coro_psf_nir.out_sr', 'cor

220 440


2026-10-30 16:43:27,417 [INFO]: [specula.simul]: Restoring: /raid1/mmenessini/calibration/WFSW/pupils/pyr_pupdata_24x24.fits
2026-10-30 16:43:27,424 [INFO]: [specula.PupData]: cupy memory used by PupData: 0.01416015625 MB
2026-10-30 16:43:27,439 [INFO]: [specula.lib.calc_geometry]: FoV reduction from 7.19 to 2.00 will be performed with a focal plane mask
/raid1/mmenessini/git/SPECULA/specula/processing_objects/ccd.py:183: RuntimeWarning: ATTENTION: emccd_gain will not be used if excess_noise is False
  warnings.warn('ATTENTION: emccd_gain will not be used if excess_noise is False',
2026-10-30 16:43:27,448 [INFO]: [specula.simul]: Restoring: /raid1/mmenessini/calibration/WFSW/pupils/pyr_pupdata_48x48.fits
2026-10-30 16:43:27,451 [INFO]: [specula.PupData]: cupy memory used by PupData: 0.05517578125 MB
2026-10-30 16:43:27,457 [INFO]: [specula.simul]: Restoring: /raid1/mmenessini/calibration/WFSW/ifunc/bmc2k_vlt_ifunc.fits


220 440


2026-10-30 16:43:36,244 [INFO]: [specula.IFunc]: cupy memory used by IFunc: 260.2939453125 MB
2026-10-30 16:43:36,244 [INFO]: [specula.simul]: Restoring: /raid1/mmenessini/calibration/WFSW/m2c/bmc2k_vlt_m2c.fits
2026-10-30 16:43:36,290 [INFO]: [specula.M2C]: cupy memory used by M2C: 12.96435546875 MB
2026-10-30 16:43:36,416 [INFO]: [pushpull]: cupy memory used by PushPullGenerator: 49.4462890625 MB
2026-10-30 16:43:36,416 [INFO]: [dm]: cupy memory used by DM: 0.39208984375 MB
2026-10-30 16:43:36,618 [INFO]: [prop]: cupy memory used by AtmoPropagation: 1.849609375 MB
2026-10-30 16:43:36,625 [INFO]: [pyr_vis]: Running unmodulated pyramid
2026-10-30 16:43:36,648 [INFO]: [pyr_vis]: cupy memory used by ModulatedPyramid: 18.94189453125 MB
2026-10-30 16:43:36,652 [INFO]: [pyr_nir]: Cached circular modulation with 24 steps, amplitude: 3.00
2026-10-30 16:43:36,664 [INFO]: [pyr_nir]: cupy memory used by ModulatedPyramid: 103.87109375 MB
2026-10-30 16:43:36,665 [INFO]: [cred_vis]: cupy memory use